# Notebook 03: Pemodelan, Hyperparameter Tuning, Evaluasi Forensik & Ekspor `.joblib` (Bab 4 & Bab 5 Laporan AM552)
**Mata Kuliah**: AM552 - Data Science of IS (Program Studi S1 Sistem Informasi, FTI UKSW)  
**Kode Kelompok**: `682024046_KEL_DS`  
**Penanggung Jawab Wilayah**: Member 1 — Arthur Willyam Liang (NIM: 682024046) [Project Lead]  
**Peran Teknis**: *Machine Learning Modeling, 5-Fold Hyperparameter Tuning, 6-Gate Anti-Leakage Forensic Audit & Deployment Pipeline*

---
## 4.1 Pilihan Model (1 Baseline Sederhana + 2 Model Alternatif) & 4.2 Hyperparameter Tuning 5-Fold
Sesuai pedoman resmi Laporan Tugas Akhir AM552 FTI UKSW, eksperimen pemodelan membandingkan:
1. **Model 0 (Dummy Baseline)**: `DummyClassifier(strategy='most_frequent')` sebagai batas bawah statistik (*statistical floor*).
2. **Model 1 (Baseline Sederhana — Bab 4.1 / Model Terpilih)**: `LogisticRegression(class_weight='balanced', C=1.0)` dengan ekstraksi fitur `TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)`.
3. **Model 2 (Alternatif 1 — Probabilistic NLP)**: `ComplementNB(alpha=0.2)` (varian Multinomial Naive Bayes untuk distribusi kelas teks yang timpang).
4. **Model 3 (Alternatif 2 — Large-Margin Classifier)**: `CalibratedClassifierCV(LinearSVC(C=0.5, class_weight='balanced'))`.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import joblib, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Muat ringkasan hasil training & tuning 5-fold serta artefak model .joblib
with open('../models/modeling_results_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)
artifact = joblib.load('../models/genai_absa_pipeline.joblib')

print('=== RINGKASAN KONFIGURASI PEMODELAN & HYPERPARAMETER TUNING (BAB 4.1 - 4.3) ===')
print(f"Model Terbaik Tersimpan  : {artifact['model_name']}")
print(f"Best Hyperparameters     : {summary['best_params']}")
print(f"5-Fold CV Macro F1-Score : {summary['best_cv_macro_f1']:.4f}")
print(f"Partisi Data Latih / Uji : {summary['train_rows_80pct']:,} baris (80%) / {summary['test_rows_20pct']:,} baris (20%)")
print(f"Langkah Pipeline (.joblib): {list(artifact['pipeline'].named_steps.keys())}")

=== RINGKASAN KONFIGURASI PEMODELAN & HYPERPARAMETER TUNING (BAB 4.1 - 4.3) ===
Model Terbaik Tersimpan  : Tuned Logistic Regression (TF-IDF 1-2 Gram, C=0.5, Balanced)
Best Hyperparameters     : {'clf__C': 0.5}
5-Fold CV Macro F1-Score : 0.6444
Partisi Data Latih / Uji : 56,976 baris (80%) / 14,245 baris (20%)
Langkah Pipeline (.joblib): ['tfidf', 'clf']


---
## 5.1 Metrik Evaluasi & 5.2 Perbandingan Hasil Model
Karena distribusi kelas di Google Play Store sangat timpang (Bintang 5 mendominasi `> 58%`, sedangkan kelas Netral Bintang 3 hanya `3,84%`), metrik **Accuracy** biasa tidak dapat dijadikan acuan tunggal. Evaluasi utama menggunakan **Macro F1-Score**, **Balanced Accuracy**, **ROC-AUC (One-vs-Rest)**, dan **Matthews Correlation Coefficient (MCC)**.

In [2]:
df_comp = pd.DataFrame(summary['model_comparison'])
print('=== TABEL PERBANDINGAN HASIL MODEL PADA DATA UJI (BAB 5.2 LAPORAN AM552) ===')
for m in summary['model_comparison']:
    print(
        f"- {m['model_name']:<48} | Train Macro-F1: {m['train_macro_f1']:.4f} | "
        f"Test Macro-F1: {m['test_macro_f1']:.4f} | Weighted-F1: {m['test_weighted_f1']:.4f} | "
        f"Balanced Acc: {m['test_balanced_acc']:.4f} | ROC-AUC: {m['test_roc_auc']:.4f} | MCC: {m['test_mcc']:.4f}"
    )

best_m = summary['best_test_metrics']
print('\n=== KEPUTUSAN PEMILIHAN MODEL TERBAIK (BAB 5.2) ===')
print(f"Model Terpilih   : {summary['best_model_name']}")
print(f"Test Macro-F1    : {best_m['macro_f1']:.4f} (TERTINGGI — Mengungguli Dummy +0.3736, ComplementNB +0.0694, LinearSVC +0.0513)")
print(f"Test Balanced Acc: {best_m['balanced_accuracy']:.4f} (TERTINGGI pada pengenalan adil lintas 3 kelas)")
print(f"Test Weighted-F1 : {best_m['weighted_f1']:.4f} | Test ROC-AUC OvR: {best_m['roc_auc_ovr']:.4f} | Test MCC: {best_m['mcc']:.4f}")

=== TABEL PERBANDINGAN HASIL MODEL PADA DATA UJI (BAB 5.2 LAPORAN AM552) ===
- 0. DummyClassifier (Most Frequent Baseline)      | Train Macro-F1: 0.2776 | Test Macro-F1: 0.2776 | Weighted-F1: 0.5941 | Balanced Acc: 0.3333 | ROC-AUC: 0.5000 | MCC: 0.0000
- 1. Logistic Regression (Tuned C=0.5)             | Train Macro-F1: 0.7380 | Test Macro-F1: 0.6502 | Weighted-F1: 0.8642 | Balanced Acc: 0.6986 | ROC-AUC: 0.8831 | MCC: 0.6745
- 2. Complement Naive Bayes (Tuned alpha=0.2)      | Train Macro-F1: 0.6363 | Test Macro-F1: 0.5803 | Weighted-F1: 0.8467 | Balanced Acc: 0.6109 | ROC-AUC: 0.8711 | MCC: 0.6882
- 3. Calibrated LinearSVC (C=0.5, Balanced Margin) | Train Macro-F1: 0.6277 | Test Macro-F1: 0.6036 | Weighted-F1: 0.8864 | Balanced Acc: 0.6122 | ROC-AUC: 0.8839 | MCC: 0.7662

=== KEPUTUSAN PEMILIHAN MODEL TERBAIK (BAB 5.2) ===
Model Terpilih   : Tuned Logistic Regression (TF-IDF 1-2 Gram, C=0.5, Balanced)
Test Macro-F1    : 0.6502 (TERTINGGI — Mengungguli Dummy +0.3736, ComplementNB +0.

---
## 5.3 Audit Forensik Metrik (`metric_sanity_audit.py`), Analisis Kesalahan & Fitur Terpenting
Menjalankan pemeriksaan **6 Gerbang Forensik Anti-Leakage** untuk membuktikan tidak ada kebocoran target (`score` atau `app_name`), menguji ketahanan pada irisan ulasan unik (*0% train overlap*) & ulasan substantif ($\ge 5$ kata), menguji generalisasi lintas aplikasi (*Leave-One-App-Out*), serta mengekstrak kata kunci pendorong prediksi (*Linear SHAP / Log-Odds Feature Attribution*).

In [3]:
import sys
sys.path.insert(0, '../05_evaluation')
import metric_sanity_audit  # type: ignore

with open('../05_evaluation/audit_results.json', 'r', encoding='utf-8') as f:
    audit_res = json.load(f)

print('=== 1. STATUS 6 GERBANG FORENSIK ANTI-LEAKAGE ===')
for k in ['audit_1_leakage', 'audit_2_imbalance_dummy', 'audit_3_duplicate_and_length_slices', 'audit_4_leave_one_app_out', 'audit_5_split_hygiene', 'audit_6_permutation_sanity']:
    print(f"  [PASS={audit_res[k]['passed']}] {audit_res[k]['audit_name']}")

print('\n=== 2. EVALUASI IRISAN DATA (DUPLICATE & SUBSTANTIVE REVIEW SLICES - AUDIT 3) ===')
for s_key, s_val in audit_res['audit_3_duplicate_and_length_slices']['slices'].items():
    sm = s_val['metrics']
    print(f"  - {s_val['slice_name']:<52} (N={s_val['n_rows']:>5,}): Macro-F1={sm['macro_f1']:.4f} | ROC-AUC={sm['roc_auc_ovr']:.4f}")

print('\n=== 3. GENERALISASI LINTAS APLIKASI (LEAVE-ONE-APP-OUT / LOAO - AUDIT 4) ===')
for loao in audit_res['audit_4_leave_one_app_out']['per_app_loao_results']:
    lm = loao['metrics']
    print(f"  - Held-Out App: {loao['held_out_app']:<26} (N_test={loao['n_test_app']:>5,}): Macro-F1={lm['macro_f1']:.4f} | ROC-AUC={lm['roc_auc_ovr']:.4f}")

print('\n=== 4. TOP FITUR PENDORONG SENTIMEN (FEATURE ATTRIBUTION / LINEAR SHAP) ===')
for cls_label, feat_list in audit_res['top_features_per_class'].items():
    print(f"  * {cls_label:<22}: {', '.join(feat_list[:8])}")

=== 1. STATUS 6 GERBANG FORENSIK ANTI-LEAKAGE ===
  [PASS=True] AUDIT-1: Circular / Target Leakage Check
  [PASS=True] AUDIT-2: Class Imbalance Illusion vs. Dummy Baseline
  [PASS=True] AUDIT-3: Exact-Text Duplicate & Substantive Review (>=5 Words) Slice Audit
  [PASS=True] AUDIT-4: Leave-One-App-Out (LOAO) Cross-Domain Generalization
  [PASS=True] AUDIT-5: Pipeline Split Hygiene & Zero-ID Overlap Check
  [PASS=True] AUDIT-6A: Randomized-Label Permutation Sanity Test (y_shuffled)

=== 2. EVALUASI IRISAN DATA (DUPLICATE & SUBSTANTIVE REVIEW SLICES - AUDIT 3) ===
  - 1. Full Test Set (All Rows)                          (N=10,684): Macro-F1=0.6435 | ROC-AUC=0.8727
  - 2. Short Generic Reviews (< 5 Words)                 (N=6,058): Macro-F1=0.6144 | ROC-AUC=0.8415
  - 3. Deduplicated Unseen-Text Test Set (0% Train Overlap) (N=6,163): Macro-F1=0.6332 | ROC-AUC=0.8657
  - 4. Substantive Complex Reviews (>= 5 Words)          (N=4,626): Macro-F1=0.6353 | ROC-AUC=0.8705

=== 3. GENERALISASI LIN

---
## 5.4 Pencapaian Ukuran Keberhasilan (Bab 1.4) & 5.5 Verifikasi Konsistensi Artefak `.joblib` dengan Prototype Bab 6
Memvalidasi ketercapaian seluruh target kuantitatif yang ditetapkan pada Bab 1.4 serta menguji kesesuaian inferensi model `.joblib` dengan modul `prototype/core_predictor.py`.

In [4]:
sys.path.insert(0, '../prototype')
from core_predictor import predict_single_review  # type: ignore

print('=== EVALUASI PENCAPAIAN UKURAN KEBERHASILAN PROYEK (BAB 1.4 vs BAB 5.4) ===')
targets = [
    ('1. Test Macro F1-Score >= 0.63', best_m['macro_f1'], 0.63, best_m['macro_f1'] >= 0.63),
    ('2. Test ROC-AUC (OvR) >= 0.85', best_m['roc_auc_ovr'], 0.85, best_m['roc_auc_ovr'] >= 0.85),
    ('3. Gain Macro-F1 di atas Dummy Baseline >= +0.35', best_m['macro_f1'] - 0.2776, 0.35, (best_m['macro_f1'] - 0.2776) >= 0.35),
]
for name, actual, thresh, ok in targets:
    status_str = 'TERCAPAI (PASSED)' if ok else 'BELUM TERCAPAI'
    print(f'  - {name:<48}: Aktual = {actual:.4f} (Target >= {thresh:.2f}) -> [{status_str}]')

print('\n=== UJI KONSISTENSI INFERENSI PIPELINE .JOBLIB UNTUK PROTOTYPE STREAMLIT (BAB 6) ===')
sample_reviews = [
    ('ChatGPT (OpenAI)', 5, 'Aplikasinya sangat pintar dan cepat, bahasa Indonesianya luwes membantu tugas coding skripsi saya!'),
    ('DeepSeek - AI Assistant', 5, 'Saya kasih bintang 5 biar dibaca developer, server sibuk terus gagal login dan jawabannya makin ngawur!'),
]
for app_n, star_r, txt in sample_reviews:
    pred = predict_single_review(txt, star_rating=star_r, app_name=app_n, artifact=artifact)
    asp_codes = [a['aspect_code'] for a in pred['detected_aspects']]
    print(f'  * [{app_n} | Bintang {star_r}] Teks: {txt[:70]}...')
    print(f"    -> Prediksi: {pred['predicted_sentiment']} (Confidence: {pred['confidence']*100:.1f}%) | Aspek: {asp_codes} | Discrepancy: {pred['is_discrepancy']}")

=== EVALUASI PENCAPAIAN UKURAN KEBERHASILAN PROYEK (BAB 1.4 vs BAB 5.4) ===
  - 1. Test Macro F1-Score >= 0.63                  : Aktual = 0.6502 (Target >= 0.63) -> [TERCAPAI (PASSED)]
  - 2. Test ROC-AUC (OvR) >= 0.85                   : Aktual = 0.8831 (Target >= 0.85) -> [TERCAPAI (PASSED)]
  - 3. Gain Macro-F1 di atas Dummy Baseline >= +0.35: Aktual = 0.3726 (Target >= 0.35) -> [TERCAPAI (PASSED)]

=== UJI KONSISTENSI INFERENSI PIPELINE .JOBLIB UNTUK PROTOTYPE STREAMLIT (BAB 6) ===
  * [ChatGPT (OpenAI) | Bintang 5] Teks: Aplikasinya sangat pintar dan cepat, bahasa Indonesianya luwes membant...
    -> Prediksi: Positif (Confidence: 92.2%) | Aspek: ['A1_Akurasi_Kecerdasan', 'A3_Performa_UI_Server', 'A4_Bahasa_Indonesia'] | Discrepancy: False
  * [DeepSeek - AI Assistant | Bintang 5] Teks: Saya kasih bintang 5 biar dibaca developer, server sibuk terus gagal l...
    -> Prediksi: Negatif (Confidence: 68.2%) | Aspek: ['A1_Akurasi_Kecerdasan', 'A3_Performa_UI_Server'] | Discrepancy: Tr